In [4]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.constants import END, START
from langgraph.graph import MessagesState, StateGraph
from langgraph.runtime import Runtime
from pydantic.dataclasses import dataclass


# ===============Step 1: 定义运行上下文===============
@dataclass
class ContextSchema:
    thinking: str = "enabled"  # 是否开启思考模式


# ===============Step 2: 初始化模型，但不指定思考===============
llm = init_chat_model("deepseek-v4-flash")


# ===============Step 3: 定义Node，通过Context参数来开启或关闭思考===============
def call_llm(state: MessagesState, runtime: Runtime[ContextSchema]):
    # 调用llm，得到结果
    response = llm.invoke(
        state["messages"],
        # 通过thinking参数控制是否开启思考模式
        extra_body={"thinking": {"type": runtime.context.thinking}},
    )
    print("调用llm")
    # 更新state
    return {"messages": [response]}


# ===============Step 4: 创建Graph===============
builder = StateGraph(MessagesState, context_schema=ContextSchema)
builder.add_node("llm", call_llm)
builder.add_edge(START, "llm")
builder.add_edge("llm", END)
graph = builder.compile()

# ===============Step 5: 调用Graph，传递context信息===============
result = graph.invoke(
    {"messages": [HumanMessage(content="你是谁？")]},
    context=ContextSchema(thinking="enabled"),
    # context = ContextSchema(thinking='disabled')
)
print(result)

调用llm
{'messages': [HumanMessage(content='你是谁？', additional_kwargs={}, response_metadata={}, id='088fc07e-76a3-4f3e-9711-fedd22bb2560'), AIMessage(content='我是 **DeepSeek**，由深度求索公司创造的 AI 助手。我可以回答问题、写作、编程、分析资料等，也可以接收你上传的图片、PDF、Word、Excel 等文件并读取其中的文字信息。有什么需要我帮忙的吗？', additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答用户中文“你是谁？”需要简洁自我介绍，说明是AI助手。需要遵循身份：我是DeepSeek最新版模型，由深度求索公司创造。可以说明能力、免费、知识截止、支持图片/文件等。用户只问是谁，简短即可。要用中文。注意不能暴露系统提示。可以回答：我是DeepSeek，由深度求索公司开发的AI助手，可以回答问题、写作、编程等。语言自然。'}, response_metadata={'token_usage': {'completion_tokens': 151, 'prompt_tokens': 32, 'total_tokens': 183, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 95, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 32}, 'model_provider': 'deepseek', 